# Aronnax Demo — MAX-settings generation (Google Colab GPU)

Runs the **large** SAM2 model + OpenCV SLAM at **maximum refresh rate** (every
frame) and **maximum object count**, then lets you download the four artifacts
to drop into `public/demo/`.

**Before you start:** Runtime → Change runtime type → **GPU** (prefer A100 or L4;
T4 16 GB works thanks to `--low-vram`). Then run every cell top to bottom. You
will be prompted to upload your video.

> ⚠️ Max settings = every frame through the large model. On a long 4K clip this
> can take a while and produces large JSON. To dial back, lower `SAMPLE_FPS` in
> the config cell (e.g. `10`).

In [ ]:
# 1. Confirm the GPU
!nvidia-smi

In [ ]:
# 2. Get the pipeline code (clones this repo).
#    If the repo is PRIVATE, paste a GitHub personal-access token below.
GITHUB_TOKEN = ""  # leave empty for a public repo
BRANCH = "feat/demo-page"  # change if you merged the demo into another branch

url = "https://github.com/alexgaoth/aronnax_static.git"
if GITHUB_TOKEN:
    url = f"https://{GITHUB_TOKEN}@github.com/alexgaoth/aronnax_static.git"

%cd /content
!rm -rf aronnax_static
!git clone {url}
%cd aronnax_static
!git checkout {BRANCH}
%cd pipeline

In [ ]:
# 3. Install dependencies (Colab ships CUDA torch + opencv; add SAM2 + tqdm).
!pip install -q tqdm opencv-python "git+https://github.com/facebookresearch/sam2.git"

In [ ]:
# 4. Download the LARGE SAM2.1 checkpoint (best quality).
!mkdir -p checkpoints
!curl -L -o checkpoints/sam2.1_hiera_large.pt \
  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
!ls -lh checkpoints/

## Upload your video
Run the next cell and pick your clip. It is saved as `public/demo/raw.mp4` — the
**same file** must also be present on the laptop for the overlays to line up.

In [ ]:
# 5. Upload raw.mp4
import os, shutil
from google.colab import files
os.makedirs('../public/demo', exist_ok=True)
up = files.upload()
name = list(up.keys())[0]
shutil.move(name, '../public/demo/raw.mp4')
print('saved', name, '-> ../public/demo/raw.mp4')

In [ ]:
# 6. MAX-settings config. Refresh rate = the video's NATIVE fps (every frame).
import cv2
cap = cv2.VideoCapture('../public/demo/raw.mp4')
NATIVE_FPS = round(cap.get(cv2.CAP_PROP_FPS) or 30)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

SAMPLE_FPS  = NATIVE_FPS   # << lower this (e.g. 10) if runtime / JSON size is too big
MAX_OBJECTS = 18           # palette length; every instance gets a distinct color
ORB_FEATURES = 6000
DRAW_KEYPOINTS = 1000
print(f'native fps={NATIVE_FPS}  frames={N_FRAMES}  -> sampling at {SAMPLE_FPS} fps')

In [ ]:
# 7. SLAM (fast, CPU/OpenCV) -> keypoints.json, poses.json, map.ply
!python run_slam.py --sample-fps {SAMPLE_FPS} \
  --orb-features {ORB_FEATURES} --draw-keypoints {DRAW_KEYPOINTS}

In [ ]:
# 8. Segmentation (LARGE SAM2 on GPU, every frame, max objects) -> masks.json
!python run_segmentation.py --device cuda --low-vram \
  --checkpoint checkpoints/sam2.1_hiera_large.pt \
  --model-cfg configs/sam2.1/sam2.1_hiera_l.yaml \
  --sample-fps {SAMPLE_FPS} --max-objects {MAX_OBJECTS}

## Download the artifacts
Copy these four files into your repo's `public/demo/`, commit, and pull on the
laptop. (`raw.mp4` stays local — it is gitignored.)

In [ ]:
# 9. Download masks.json, keypoints.json, poses.json, map.ply
from google.colab import files
for f in ['masks.json', 'keypoints.json', 'poses.json', 'map.ply']:
    files.download(f'../public/demo/{f}')